<a href="https://colab.research.google.com/github/nadiyasoaib/E-commerce-product-pricing/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Amazon-ML-Challenge-2025')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
import os

# Explicitly add the project root to sys.path
PROJECT_ROOT = '/content/drive/MyDrive/Amazon-ML-Challenge-2025'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Also ensure current working directory is set (though sys.path is more robust for imports)
os.chdir(PROJECT_ROOT)

In [ ]:
%pip install -U pandas scikit-learn torch torchvision sentence-transformers timm tqdm Pillow requests psutil

# This command installs the specific CLIP library from its GitHub repository
%pip install git+https://github.com/openai/CLIP.git

print("\nAll libraries installed or updated.")
print("IMPORTANT: Please RESTART THE KERNEL now for the changes to take effect")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-fijzx2bt
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-fijzx2bt
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 949.0 kB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=73f4234f724b6991409c89bd4aa171fc43c90e7eb50972ba33377d2e15375746
  Stored in directory: /tmp/pip-ephem-wheel-cache-q66rwnvn/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip

All libraries installed or updated.
IMPORTANT: Please RESTART THE KERNEL now for the changes to take effect


In [ ]:
# Python Libraries
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from tqdm.notebook import tqdm
import warnings

# ML Libraries
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import timm
from src.utils import download_images # Make sure utils.py is in src/ folder

# Configuration
warnings.filterwarnings('ignore')

TRAIN_IMAGE_DIR = 'test_images'  # Folder with TRAIN photos
TEST_IMAGE_DIR = 'train_images'   # Folder with TEST photos


# Device Selection for Mac M1 GPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Setup Complete. Using device: {DEVICE}")

Setup Complete. Using device: cpu


In [ ]:
import os
import sys

print(f"Current Working Directory: {os.getcwd()}")
print("Python Path (sys.path):")
for p in sys.path:
    print(f"  - {p}")

expected_src_path = os.path.join(os.getcwd(), 'src')
print(f"\nChecking for 'src' directory at: {expected_src_path}")
if os.path.exists(expected_src_path):
    print("✅ 'src' directory found!")
    expected_utils_path = os.path.join(expected_src_path, 'utils.py')
    print(f"Checking for 'utils.py' at: {expected_utils_path}")
    if os.path.exists(expected_utils_path):
        print("✅ 'src/utils.py' found!")
        print("If both are found, please ensure your kernel has been restarted after any `pip install` commands, and then re-run the cell with the import statement.")
    else:
        print("❌ 'src/utils.py' not found inside the 'src' directory. Please check the file name or location.")
else:
    print("❌ 'src' directory not found in the current working directory. Please ensure your 'src' folder is at the project root or adjust `sys.path` accordingly.")



Current Working Directory: /content/drive/MyDrive/Amazon-ML-Challenge-2025
Python Path (sys.path):
  - /content
  - /env/python
  - /usr/lib/python312.zip
  - /usr/lib/python3.12
  - /usr/lib/python3.12/lib-dynload
  - 
  - /usr/local/lib/python3.12/dist-packages
  - /usr/lib/python3/dist-packages
  - /usr/local/lib/python3.12/dist-packages/IPython/extensions
  - /root/.ipython
  - /content/drive/MyDrive/Amazon-ML-Challenge-2025

Checking for 'src' directory at: /content/drive/MyDrive/Amazon-ML-Challenge-2025/src
✅ 'src' directory found!
Checking for 'utils.py' at: /content/drive/MyDrive/Amazon-ML-Challenge-2025/src/utils.py
✅ 'src/utils.py' found!
If both are found, please ensure your kernel has been restarted after any `pip install` commands, and then re-run the cell with the import statement.


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU (CUDA) is available.")
    print(f"Current device: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("GPU (MPS) is available (Apple Silicon).")
elif torch.backends.npu.is_available():
    print("GPU (NPU) is available (Huawei Ascend).")
else:
    print("No GPU found. Running on CPU.")

# Also explicitly check the DEVICE variable defined earlier
print(f"The DEVICE variable is currently set to: {DEVICE}")

AttributeError: module 'torch.backends' has no attribute 'npu'

In [ ]:
import torch

if torch.cuda.is_available():
    print("torch.cuda.is_available() returns True. A CUDA GPU is detected.")
    print(f"Current CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("torch.cuda.is_available() returns False. No CUDA GPU detected.")


torch.cuda.is_available() returns False. No CUDA GPU detected.


In [ ]:
# Load
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

print(f"Train data shape: {df_train.shape}")
print(f"Test data shape: {df_test.shape}")

print("\nTrain data sample:")
df_train.head()

Train data shape: (47698, 4)
Test data shape: (35952, 3)

Train data sample:


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [ ]:
import pandas as pd
# Cell for FAST and EFFICIENT Downloading

# Import the main download function from your updated utils.py
from src.utils import download_images
from pathlib import Path

def run_smart_download(df, img_dir):
    """
    Checks for missing files and calls the official download script only for them.
    """
    img_dir_path = Path(img_dir)
    print(f"--- Verifying images in '{img_dir_path.name}' ---")

    expected_ids = set(df['sample_id'].astype(str))
    existing_ids = {f.stem for f in img_dir_path.glob('*.jpg')}
    missing_ids = expected_ids - existing_ids

    print(f"Found {len(existing_ids)} of {len(expected_ids)} expected images.")

    if not missing_ids:
        print("All images are present. No download needed.")
        return

    print(f"{len(missing_ids)} missing image(s) detected. Preparing to download.")

    # Filter the DataFrame to get only the rows for the missing images
    df_missing = df[df['sample_id'].astype(str).isin(missing_ids)]

    # Create the list of tasks [(sample_id, image_link), ...]
    tasks_to_run = list(zip(df_missing['sample_id'], df_missing['image_link']))

    # Call the download function from utils.py
    download_images(tasks_to_run, str(img_dir_path))

    print("\nDownload attempt complete.")

# --- Load the full dataframes first to ensure slicing is correct ---
df_train_full = pd.read_csv('train.csv')
df_test_full = pd.read_csv('test.csv')

# --- Slice the dataframes to 20k images ---
df_train_subset = df_train_full.head(20000)
df_test_subset = df_test_full.head(10000)

# Execute the download for both sets using the subsets.
# Make sure TRAIN_IMAGE_DIR and TEST_IMAGE_DIR are set correctly from your setup cell!
run_smart_download(df_train_subset, TRAIN_IMAGE_DIR)
print("-" * 30)
run_smart_download(df_test_subset, TEST_IMAGE_DIR)

--- Verifying images in 'test_images' ---
Found 21454 of 20000 expected images.
All images are present. No download needed.
------------------------------
--- Verifying images in 'train_images' ---
Found 10000 of 10000 expected images.
All images are present. No download needed.


In [ ]:
import pandas as pd
from pathlib import Path # <-- IMPORT THE PATH OBJECT
from tqdm import tqdm
import warnings

# Suppress unnecessary warnings
warnings.filterwarnings('ignore')

# Function to Create Directories
def create_directories(*dir_paths):
    """Creates one or more directories if they do not already exist."""
    print("Ensuring image directories exist...")
    for path in dir_paths:
        # The path variable is now a proper Path object
        path.mkdir(parents=True, exist_ok=True)
    print("All necessary directories are created or already exist.")

# Setup
# Define base data directory and load dataframes
BASE_PATH = Path('.') # Changed this from '../src' to '.'

# THE FIX: Use Path() to define your directories
TRAIN_IMAGE_DIR = BASE_PATH / 'test_images'
TEST_IMAGE_DIR = BASE_PATH / 'train_images'

# Call the function to create the directories
create_directories(TRAIN_IMAGE_DIR, TEST_IMAGE_DIR)

# Load DataFrames
# Assuming df_test is loaded from a CSV file in the 'data' folder
df_train = pd.read_csv('train.csv') # Removed '../'
df_test = pd.read_csv('test.csv')   # Removed '../'

# Clean the Test Images Folder
print(f"\nCleaning the test images folder: {TEST_IMAGE_DIR}")

# Create a set of valid sample IDs for the test set for fast lookups
valid_test_ids = set(df_test['sample_id'].astype(str))

files_to_delete = []
# Find all files that do NOT belong in the test set
for f_path in TEST_IMAGE_DIR.glob('*.jpg'):
    if f_path.stem not in valid_test_ids:
        files_to_delete.append(f_path)

if not files_to_delete:
    print("No extra files found. Folder is already clean!")
else:
    print(f"Found {len(files_to_delete)} extra files to delete. Deleting now...")
    for f in tqdm(files_to_delete, desc="Cleaning"):
        f.unlink() # This deletes the file
    print("Cleaning complete.")

# Final Verification
train_count = len(list(TRAIN_IMAGE_DIR.glob('*.jpg')))
test_count = len(list(TEST_IMAGE_DIR.glob('*.jpg')))

print("\nFinal File Counts")
print(f"Images in TRAIN folder ({TRAIN_IMAGE_DIR.name}): {train_count}")
print(f"Images in TEST folder ({TEST_IMAGE_DIR.name}):  {test_count}")

Ensuring image directories exist...
All necessary directories are created or already exist.

Cleaning the test images folder: train_images
No extra files found. Folder is already clean!

Final File Counts
Images in TRAIN folder (test_images): 21453
Images in TEST folder (train_images):  10000


In [ ]:
# Cell 7: Parse Catalog Content
import re

def parse_content(content_string):
    """
    Parses the raw catalog_content string into separate, clean features.
    """
    if not isinstance(content_string, str):
        content_string = ""

    lines = content_string.strip().split('\n')

    # Default values
    item_name = ""
    bullet_points = []
    prod_desc = ""
    value = 1.0  # Default to 1 if not found
    unit = "Unknown"

    for line in lines:
        if line.lower().startswith("item name:"):
            item_name = line[len("item name:"):].strip()
        elif line.lower().startswith("bullet point"):
            bp_text = re.sub(r'Bullet Point \d+:', '', line, flags=re.IGNORECASE).strip()
            bullet_points.append(bp_text)
        elif line.lower().startswith("product description:"):
            prod_desc = line[len("product description:"):].strip()
        elif line.lower().startswith("value:"):
            try:
                value = float(line[len("value:"):].strip())
            except (ValueError, TypeError):
                value = 1.0 # Keep default if parsing fails
        elif line.lower().startswith("unit:"):
            unit = line[len("unit:"):].strip()

    # Combine all text fields into a single 'clean_text' feature
    clean_text = " ".join([item_name] + bullet_points + [prod_desc]).strip()

    return pd.Series([clean_text, value, unit], index=['clean_text', 'quantity', 'unit'])

# --- Apply the function to both train and test dataframes ---
print("Parsing training data...")
df_train_parsed = df_train['catalog_content'].apply(parse_content)
df_train = pd.concat([df_train, df_train_parsed], axis=1)

print("Parsing test data...")
df_test_parsed = df_test['catalog_content'].apply(parse_content)
df_test = pd.concat([df_test, df_test_parsed], axis=1)

# Display the new columns to verify
print("\nNew features created successfully!")
df_train[['clean_text', 'quantity', 'unit']].head()

Parsing training data...
Parsing test data...

New features created successfully!


,clean_text,quantity,unit
0,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",72.00,Fl Oz
1,"Salerno Cookies, The Original Butter Cookies, ...",32.00,Ounce
2,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",11.40,Ounce
3,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,11.25,Ounce
4,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",12.00,Count


In [ ]:
import ssl
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import clip
from PIL import Image
import gc
import warnings
warnings.filterwarnings('ignore')

# 1. SETUP
print("\nStep 1: Setting up environment")
# Corrected DATA_DIR and image paths to point to the local Colab storage
TRAIN_IMAGE_DIR = Path('/content/test_images_local')
TEST_IMAGE_DIR = Path('/content/train_images_local')

# FIX: Correctly check for CUDA first, then MPS, then CPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

ssl._create_default_https_context = ssl._create_unverified_context
SAVE_DIR_MID = Path("embeddings_medium"); SAVE_DIR_MID.mkdir(exist_ok=True)

# Corrected paths for CSV files (these are now loaded from the global scope by the notebook)
# df_train = pd.read_csv("train.csv") # These are now loaded as subsets
# df_test = pd.read_csv("test.csv")   # These are now loaded as subsets

print(f"Using device: {DEVICE}")

# 2. LOAD THE UPGRADED MODEL
print("\nStep 2: Loading BALANCED CLIP model")
# This is a powerful model that is stable on 8GB RAM systems.
clip_model, clip_preprocess = clip.load("ViT-B/16", device=DEVICE)
print("CLIP model loaded.")

# 3. DEFINE THE DATA LOADER
# This class loads images one by one for the DataLoader.
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert("RGB")
            return self.transform(img)
        except (FileNotFoundError, OSError):
            # Return a blank placeholder if an image is missing
            return torch.zeros(3, 224, 224)

# 4. GENERATION FUNCTION
def generate_image_embeddings_stable(df, image_dir, prefix):
    print(f"\n--- Generating '{prefix}' Image Embeddings")
    image_paths = [Path(image_dir) / f"{sid}.jpg" for sid in df['sample_id'].astype(str)]

    dataset = ImageDataset(image_paths, clip_preprocess)
    # num_workers=0 is crucial for stability on macOS in a notebook
    dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0) # Increased batch_size, num_workers=0 for stability

    all_embeds = []
    with torch.no_grad():
        for image_batch in tqdm(dataloader, desc=f"Processing {prefix} batches"):
            image_batch = image_batch.to(DEVICE)
            # Generate embeddings and move them to CPU immediately to save GPU memory
            embeds = clip_model.encode_image(image_batch).cpu().numpy()
            all_embeds.append(embeds)

            # Aggressively clear memory after each batch
            del image_batch, embeds
            if DEVICE == "mps":
                torch.mps.empty_cache()
            gc.collect()

    # Combine all batch results into one final array
    full_embeddings = np.vstack(all_embeds)
    # Save the complete array to a single file
    np.save(SAVE_DIR_MID / f"{prefix}_image_embeds_full.npy", full_embeddings)
    print(f"Saved feature array to: {SAVE_DIR_MID / f'{prefix}_image_embeds_full.npy'}")

# --- 5. EXECUTE THE PROCESS ---
# Use the df_train_subset and df_test_subset created in a previous cell
generate_image_embeddings_stable(df_train_subset, TRAIN_IMAGE_DIR, prefix="train")
generate_image_embeddings_stable(df_test_subset, TEST_IMAGE_DIR, prefix="test")

# Final cleanup
del clip_model
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

print("\nUpgraded 'HD' image embeddings have been generated successfully!")


Step 1: Setting up environment
Using device: cuda

Step 2: Loading BALANCED CLIP model
CLIP model loaded.

--- Generating 'train' Image Embeddings


Processing train batches: 100%|██████████| 157/157 [01:44<00:00,  1.50it/s]


Saved feature array to: embeddings_medium/train_image_embeds_full.npy

--- Generating 'test' Image Embeddings


Processing test batches: 100%|██████████| 79/79 [00:52<00:00,  1.51it/s]


Saved feature array to: embeddings_medium/test_image_embeds_full.npy

Upgraded 'HD' image embeddings have been generated successfully!


In [ ]:
# Cell: Generate Text Embeddings

import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import torch
import gc

print("--- Step 1 of 2: Generating Text Embeddings ---")

# --- Setup ---
SAVE_DIR = Path("embeddings")
SAVE_DIR.mkdir(exist_ok=True)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# FIX: Use the subsets for consistency with image embedding generation
df_train = df_train_subset # Assuming df_train_subset is available from previous cells
df_test = df_test_subset   # Assuming df_test_subset is available from previous cells

# This re-runs the parsing logic to get the 'clean_text' column
import re
BRAND_LIST = [
    'nescafe', 'starbucks', 'keurig', 'dunkin', 'lavazza', 'peet\'s', 'folgers', 'tassimo',
    'samsung', 'apple', 'sony', 'lg', 'panasonic', 'bose', 'dell', 'hp', 'lenovo', 'acer', 'microsoft',
    'nike', 'adidas', 'under armour', 'puma', 'reebok', 'new balance', 'champion',
    'lego', 'hasbro', 'mattel', 'nerf', 'funko', 'play-doh',
    'amazonbasics', 'kirkland signature', 'great value', 'up&up', 'logitech'
]
def parse_content_with_brand(content_string):
    if not isinstance(content_string, str): content_string = ""
    lines = content_string.strip().split('\n'); item_name, bullet_points, value, unit, brand = "", [], 1.0, "Unknown", "Unknown"
    for line in lines:
        if line.lower().startswith("item name:"): item_name = line[len("item name:"):].strip()
        elif line.lower().startswith("bullet point"): bullet_points.append(re.sub(r'Bullet Point \d+:', '', line, flags=re.IGNORECASE).strip())
        elif line.lower().startswith("value:"):
            try: value = float(line[len("value:"):].strip())
            except: value = 1.0
        elif line.lower().startswith("unit:"): unit = line[len("unit:"):].strip()
    clean_text = " ".join([item_name] + bullet_points).strip()
    text_for_brand_check = (item_name + " " + (bullet_points[0] if bullet_points else "")).lower()
    for b in BRAND_LIST:
        if f' {b} ' in f' {text_for_brand_check} ': brand = b; break
    return pd.Series([clean_text, value, unit, brand], index=['clean_text', 'quantity', 'unit', 'brand'])

df_train = pd.concat([df_train, df_train['catalog_content'].apply(parse_content_with_brand)], axis=1)
df_test = pd.concat([df_test, df_test['catalog_content'].apply(parse_content_with_brand)], axis=1)


# Load Model
print("\nLoading text model...")
text_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

# Generation Function
def generate_text_embeddings(df, column, prefix):
    print(f"\nGenerating Text Embeddings for '{prefix}' set")
    texts = df[column].tolist()
    EMB_CHUNK = 10000 # Process in chunks to be memory safe

    for start in tqdm(range(0, len(texts), EMB_CHUNK), desc=f"Processing {prefix} chunks"):
        end = min(start + EMB_CHUNK, len(texts))
        batch_texts = texts[start:end]

        embeds = text_model.encode(
            batch_texts,
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        np.save(SAVE_DIR / f"{prefix}_text_embeds_{start}_{end}.npy", embeds)

# Execute
generate_text_embeddings(df_train, "clean_text", prefix="train")
generate_text_embeddings(df_test, "clean_text", prefix="test")

del text_model; gc.collect()
if DEVICE == "mps": torch.mps.empty_cache()

print("\n" + "="*60)
print("Text embeddings have been generated successfully!")
print(f"   Files are saved in the '{SAVE_DIR}' folder.")
print("   You can now re-run the 'Build Final Dataset' cell.")
print("="*60)


--- Step 1 of 2: Generating Text Embeddings ---

Loading text model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Generating Text Embeddings for 'train' set


Processing train chunks:   0%|          | 0/5 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Processing test chunks:  75%|███████▌  | 3/4 [20:15<06:46, 406.48s/it]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Processing test chunks: 100%|██████████| 4/4 [24:19<00:00, 364.92s/it]


Text embeddings have been generated successfully!
   Files are saved in the 'embeddings' folder.
   You can now re-run the 'Build Final Dataset' cell.


In [ ]:
# Cell: Build Final Datasets from All Features

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gc

print("\nStep 1: Loading all data and feature references")

# Define paths to your data and embedding folders
SAVE_DIR_TEXT = Path("embeddings") # Corrected path
SAVE_DIR_IMAGE = Path("embeddings_medium")

# Load full dataframes first to ensure slicing is correct
df_train_full = pd.read_csv("train.csv")
df_test_full = pd.read_csv("test.csv")

# Use the subsets for consistency with embedding generation
df_train = df_train_full.head(20000)
df_test = df_test_full.head(10000)

# --- This re-runs the parsing logic to ensure the brand/unit columns are available ---
# (It's very fast and safe to re-run)
BRAND_LIST = [
    'nescafe', 'starbucks', 'keurig', 'dunkin', 'lavazza', 'peet\'s', 'folgers', 'tassimo',
    'samsung', 'apple', 'sony', 'lg', 'panasonic', 'bose', 'dell', 'hp', 'lenovo', 'acer', 'microsoft',
    'nike', 'adidas', 'under armour', 'puma', 'reebok', 'new balance', 'champion',
    'lego', 'hasbro', 'mattel', 'nerf', 'funko', 'play-doh',
    'amazonbasics', 'kirkland signature', 'great value', 'up&up', 'logitech'
]
import re
def parse_content_with_brand(content_string):
    if not isinstance(content_string, str): content_string = ""
    lines = content_string.strip().split('\n'); item_name, bullet_points, value, unit, brand = "", [], 1.0, "Unknown", "Unknown"
    for line in lines:
        if line.lower().startswith("item name:"): item_name = line[len("item name:"):].strip()
        elif line.lower().startswith("bullet point"): bullet_points.append(re.sub(r'Bullet Point \d+:', '', line, flags=re.IGNORECASE).strip())
        elif line.lower().startswith("value:"):
            try: value = float(line[len("value:"):].strip())
            except: value = 1.0
        elif line.lower().startswith("unit:"): unit = line[len("unit:"):].strip()
    clean_text = " ".join([item_name] + bullet_points).strip()
    text_for_brand_check = (item_name + " " + (bullet_points[0] if bullet_points else "")).lower()
    for b in BRAND_LIST:
        if f' {b} ' in f' {text_for_brand_check} ': brand = b; break
    return pd.Series([clean_text, value, unit, brand], index=['clean_text', 'quantity', 'unit', 'brand'])

df_train = pd.concat([df_train, df_train['catalog_content'].apply(parse_content_with_brand)], axis=1)
df_test = pd.concat([df_test, df_test['catalog_content'].apply(parse_content_with_brand)], axis=1)
print("Dataframes with brand/quantity features are ready.")


# --- Step 2: Prepare categorical features ---
print("\nStep 2: Preparing categorical features (unit and brand)")
train_cats = pd.get_dummies(df_train[['unit', 'brand']], prefix=['unit', 'brand'])
test_cats = pd.get_dummies(df_test[['unit', 'brand']], prefix=['unit', 'brand'])
train_cats_aligned, test_cats_aligned = train_cats.align(test_cats, join='outer', axis=1, fill_value=0)
print("Categorical features prepared.")


# Step 3: Combine and save the final arrays
def build_final_dataset(prefix, df, cat_features):
    print(f"\nBuilding final dataset for '{prefix}'")

    # Load all text embedding chunks and combine them
    text_files = sorted(SAVE_DIR_TEXT.glob(f"{prefix}_text_embeds_*.npy"))
    if not text_files: raise FileNotFoundError(f"No text embedding files found for '{prefix}'.")
    text_embeds = np.vstack([np.load(f) for f in text_files])

    # ENSURE text_embeds matches the current df size
    if text_embeds.shape[0] > df.shape[0]:
        print(f"Warning: Truncating {prefix} text embeddings from {text_embeds.shape[0]} to {df.shape[0]} rows.")
        text_embeds = text_embeds[:df.shape[0]]
    elif text_embeds.shape[0] < df.shape[0]:
        raise ValueError(f"Error: {prefix} text embeddings ({text_embeds.shape[0]} rows) are smaller than dataframe ({df.shape[0]} rows). Please regenerate embeddings from the correct subset.")

    # Load the single, complete image embedding file
    image_embeds = np.load(SAVE_DIR_IMAGE / f"{prefix}_image_embeds_full.npy")

    # ENSURE image_embeds matches the current df size
    if image_embeds.shape[0] > df.shape[0]:
        print(f"Warning: Truncating {prefix} image embeddings from {image_embeds.shape[0]} to {df.shape[0]} rows.")
        image_embeds = image_embeds[:df.shape[0]]
    elif image_embeds.shape[0] < df.shape[0]:
        raise ValueError(f"Error: {prefix} image embeddings ({image_embeds.shape[0]} rows) are smaller than dataframe ({df.shape[0]} rows). Please regenerate embeddings from the correct subset.")

    # Get the quantity column and fill any missing values
    quantity = df['quantity'].fillna(1.0).values.reshape(-1, 1)

    # Get the one-hot encoded categorical features
    cats = cat_features.values

    # This is the key step: stack everything side-by-side
    final_X = np.hstack([
        text_embeds.astype(np.float32),      # Text features
        image_embeds.astype(np.float32),     # Image features
        quantity.astype(np.float32),         # Quantity feature
        cats.astype(np.float32)              # Brand and Unit features
    ])

    # Save the final, combined array
    save_path = SAVE_DIR_IMAGE / f"final_X_{prefix}_medium_with_brand.npy"
    np.save(save_path, final_X)
    print(f"Saved final feature array to: {save_path} with shape {final_X.shape}")

    # Clean up memory
    del text_embeds, image_embeds, quantity, cats, final_X; gc.collect()

# Execute for both train and test sets --
build_final_dataset("train", df_train, train_cats_aligned)
build_final_dataset("test", df_test, test_cats_aligned)

print("\nAll final feature arrays have been created successfully!")


Step 1: Loading all data and feature references
Dataframes with brand/quantity features are ready.

Step 2: Preparing categorical features (unit and brand)
Categorical features prepared.

Building final dataset for 'train'
Saved final feature array to: embeddings_medium/final_X_train_medium_with_brand.npy with shape (20000, 983)

Building final dataset for 'test'
Saved final feature array to: embeddings_medium/final_X_test_medium_with_brand.npy with shape (10000, 983)

All final feature arrays have been created successfully!


In [ ]:
import shutil
import os
from pathlib import Path

print("Copying image directories from Google Drive to local Colab storage for faster access...")

# Define source and destination paths
# Current working directory is /content/drive/MyDrive/Amazon-ML-Challenge-2025
source_train_dir = Path('./test_images')
source_test_dir = Path('./train_images')

# Local Colab runtime paths
destination_train_dir = Path('/content/test_images_local')
destination_test_dir = Path('/content/train_images_local')

# Ensure destination directories exist, clean if necessary
if destination_train_dir.exists():
    print(f"Cleaning existing local directory: {destination_train_dir}")
    shutil.rmtree(destination_train_dir)
if destination_test_dir.exists():
    print(f"Cleaning existing local directory: {destination_test_dir}")
    shutil.rmtree(destination_test_dir)

# Copy directories
print(f"Copying '{source_train_dir}' to '{destination_train_dir}' (this might take a few minutes for many files)...")
shutil.copytree(source_train_dir, destination_train_dir)
print(f"Copying '{source_test_dir}' to '{destination_test_dir}' (this might take a few minutes for many files)...")
shutil.copytree(source_test_dir, destination_test_dir)

print("Image directories are now available on local disk for faster processing.")


Copying image directories from Google Drive to local Colab storage for faster access...
Copying 'test_images' to '/content/test_images_local' (this might take a few minutes for many files)...
Copying 'train_images' to '/content/train_images_local' (this might take a few minutes for many files)...
Image directories are now available on local disk for faster processing.


In [ ]:
# OPTIMIZED MLP FUSION FOR TEXT + IMAGE EMBEDDINGS

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from tqdm import tqdm
import gc
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("MLP FUSION: Text + Image Embeddings")
print("="*70)

# ADVANCED MLP ARCHITECTURE

class MultimodalFusionMLP(nn.Module):
    """
    Advanced fusion with separate encoders and attention.
    """
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=512, dropout=0.3):
        super().__init__()

        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.other_encoder = nn.Sequential( # Encoder for quantity, brand, etc.
            nn.Linear(other_dim, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(dropout * 0.5)
        )

        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, dropout=0.1, batch_first=True)

        # Fusion layers to combine all encoded parts
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 64, hidden_dim * 2), nn.LayerNorm(hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout * 0.7),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, text_emb, image_emb, other_emb):
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)

        # Cross-attention: text attends to image
        attended, _ = self.attention(text_enc.unsqueeze(1), image_enc.unsqueeze(1), image_enc.unsqueeze(1))
        attended = attended.squeeze(1)

        # Concatenate attended text, original image, and other features
        fused = torch.cat([attended, image_enc, other_enc], dim=1)
        output = self.fusion(fused)
        return output

# TRAINING FUNCTION (UPDATED FOR 3 INPUTS)

def train_mlp_fusion(X_text_tr, X_image_tr, X_other_tr, y_tr,
                     X_text_val, X_image_val, X_other_val, y_val,
                     epochs=100, batch_size=256, lr=5e-4):
    # FIX: Use CUDA if available, otherwise MPS, otherwise CPU
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    print(f"  Using device: {device}")

    text_dim, image_dim, other_dim = X_text_tr.shape[1], X_image_tr.shape[1], X_other_tr.shape[1]

    model = MultimodalFusionMLP(text_dim, image_dim, other_dim).to(device)

    def pseudo_huber_loss(pred, target, delta=1.0):
        residual = pred - target
        return torch.mean(delta**2 * (torch.sqrt(1 + (residual/delta)**2) - 1))

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

    train_dataset = torch.utils.data.TensorDataset(torch.FloatTensor(X_text_tr), torch.FloatTensor(X_image_tr), torch.FloatTensor(X_other_tr), torch.FloatTensor(y_tr).unsqueeze(1))
    val_dataset = torch.utils.data.TensorDataset(torch.FloatTensor(X_text_val), torch.FloatTensor(X_image_val), torch.FloatTensor(X_other_val), torch.FloatTensor(y_val).unsqueeze(1))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    best_val_loss = float('inf'); patience, patience_counter = 15, 0

    for epoch in range(epochs):
        model.train(); train_loss = 0
        for text_b, image_b, other_b, y_b in train_loader:
            text_b, image_b, other_b, y_b = text_b.to(device), image_b.to(device), other_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            output = model(text_b, image_b, other_b)
            loss = pseudo_huber_loss(output, y_b)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            train_loss += loss.item()

        model.eval(); val_loss = 0
        with torch.no_grad():
            for text_b, image_b, other_b, y_b in val_loader:
                text_b, image_b, other_b, y_b = text_b.to(device), image_b.to(device), other_b.to(device), y_b.to(device)
                output = model(text_b, image_b, other_b)
                val_loss += pseudo_huber_loss(output, y_b).item()

        train_loss /= len(train_loader); val_loss /= len(val_loader); scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss, best_model_state, patience_counter = val_loss, model.state_dict(), 0
        else:
            patience_counter += 1

        if patience_counter >= patience: print(f"    Early stopping at epoch {epoch+1}"); break
        if (epoch + 1) % 10 == 0: print(f"    Epoch {epoch+1}: train_loss={train_loss:.5f}, val_loss={val_loss:.5f}")

    model.load_state_dict(best_model_state)
    return model

# PREDICTION FUNCTION (UPDATED FOR 3 INPUTS)

def predict_mlp(model, X_text, X_image, X_other, batch_size=256):
    device = next(model.parameters()).device
    model.eval(); predictions = []
    with torch.no_grad():
        for i in tqdm(range(0, len(X_text), batch_size), desc="Predicting", leave=False):
            end_idx = min(i + batch_size, len(X_text))
            text_b = torch.FloatTensor(X_text[i:end_idx]).to(device)
            image_b = torch.FloatTensor(X_image[i:end_idx]).to(device)
            other_b = torch.FloatTensor(X_other[i:end_idx]).to(device)
            output = model(text_b, image_b, other_b)
            predictions.append(output.cpu().numpy())
    return np.vstack(predictions).flatten()

# CORRECTED DATA LOADING AND SLICING
print("\n[1/4] Loading and slicing combined embeddings...")

# Define the base project directory where 'train.csv', 'test.csv' and embedding folders are located
BASE_PROJECT_PATH = Path('/content/drive/MyDrive/Amazon-ML-Challenge-2025')

df_train = pd.read_csv(BASE_PROJECT_PATH / 'train.csv')
y_train_log = np.log1p(df_train['price'].values)

# Define paths to your data and embedding folders
# These must match where the files were *saved* by cell 8ae7d5fa.
SAVE_DIR_IMAGE = BASE_PROJECT_PATH / "embeddings_medium"
SAVE_DIR_TEXT = BASE_PROJECT_PATH / "embeddings"

# Load the SINGLE, COMBINED feature files , use correct paths
X_train_full = np.load(SAVE_DIR_IMAGE / "final_X_train_medium_with_brand.npy", allow_pickle=False)
X_test_full = np.load(SAVE_DIR_IMAGE / "final_X_test_medium_with_brand.npy", allow_pickle=False)

# Define the dimensions of your features
text_dim = 384 # From SentenceTransformer
image_dim = 512 # From ViT-B/16

# Slice the combined arrays into their constituent parts
train_text = X_train_full[:, :text_dim]
train_image = X_train_full[:, text_dim:text_dim+image_dim]
train_other = X_train_full[:, text_dim+image_dim:]

test_text = X_test_full[:, :text_dim]
test_image = X_test_full[:, text_dim:text_dim+image_dim]
test_other = X_test_full[:, text_dim+image_dim:]

print(f"✓ Text: train{train_text.shape}, test{test_text.shape}")
print(f"✓ Image: train{train_image.shape}, test{test_image.shape}")
print(f"✓ Other: train{train_other.shape}, test{test_other.shape}")
del X_train_full, X_test_full; gc.collect()

# SCALE FEATURES

print("\n[2/4] Scaling features...")
text_scaler, image_scaler, other_scaler = RobustScaler(), RobustScaler(), RobustScaler()

train_text_scaled = text_scaler.fit_transform(train_text); test_text_scaled = text_scaler.transform(test_text)
train_image_scaled = image_scaler.fit_transform(train_image); test_image_scaled = image_scaler.transform(test_image)
train_other_scaled = other_scaler.fit_transform(train_other); test_other_scaled = other_scaler.transform(test_other)

print("✓ Features scaled")

# ✅ SAVE SCALERS HERE
import joblib

joblib.dump(text_scaler, "text_scaler.pkl")
joblib.dump(image_scaler, "image_scaler.pkl")
joblib.dump(other_scaler, "other_scaler.pkl")
del train_text, train_image, train_other, test_text, test_image, test_other; gc.collect()

# K-FOLD TRAINING

print("\n[3/4] Training MLP with K-Fold...")
N_FOLDS = 5; kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_text_scaled)); test_preds = np.zeros(len(test_text_scaled))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_text_scaled), 1):
    print(f"\n{'─'*70}\nFOLD {fold}/{N_FOLDS}")

    model = train_mlp_fusion(
        train_text_scaled[train_idx], train_image_scaled[train_idx], train_other_scaled[train_idx], y_train_log[train_idx],
        train_text_scaled[val_idx], train_image_scaled[val_idx], train_other_scaled[val_idx], y_train_log[val_idx]
    )

    # 🔥 ADD HERE
    torch.save(model.state_dict(), f"model_fold_{fold}.pth")

    oof_preds[val_idx] = predict_mlp(model, train_text_scaled[val_idx], train_image_scaled[val_idx], train_other_scaled[val_idx])
    test_preds += predict_mlp(model, test_text_scaled, test_image_scaled, test_other_scaled) / N_FOLDS

    val_pred_price = np.expm1(oof_preds[val_idx]); val_actual_price = np.expm1(y_train_log[val_idx])
    fold_smape = np.mean(2 * np.abs(val_pred_price - val_actual_price) / (np.abs(val_actual_price) + np.abs(val_pred_price) + 1e-8)) * 100
    print(f"Fold {fold} SMAPE: {fold_smape:.4f}%")

    del model; gc.collect(); torch.mps.empty_cache() if torch.backends.mps.is_available() else None


# FINAL EVALUATION

print("\n[4/4] Final evaluation and submission...")
oof_prices = np.expm1(oof_preds); actual_prices = df_train['price'].values
overall_smape = np.mean(2 * np.abs(oof_prices - actual_prices) / (np.abs(actual_prices) + np.abs(oof_prices) + 1e-8)) * 100
print("\n" + "="*70 + f"\nFINAL OOF SMAPE: {overall_smape:.4f}%\n" + "="*70)

final_predictions = np.expm1(test_preds); final_predictions = np.clip(final_predictions, 0.01, None)
df_test = pd.read_csv(BASE_PROJECT_PATH / 'test.csv') # Corrected path for test.csv
submission = pd.DataFrame({'sample_id': df_test['sample_id'],'price': final_predictions})
submission.to_csv('test_out.csv', index=False)

print("\nSubmission created: test_out.csv")
print("\nFirst 10 predictions:"); print(submission.head(10))

MLP FUSION: Text + Image Embeddings

[1/4] Loading and slicing combined embeddings...
✓ Text: train(20000, 384), test(10000, 384)
✓ Image: train(20000, 512), test(10000, 512)
✓ Other: train(20000, 87), test(10000, 87)

[2/4] Scaling features...
✓ Features scaled

[3/4] Training MLP with K-Fold...

──────────────────────────────────────────────────────────────────────
FOLD 1/5
  Using device: cuda
    Epoch 10: train_loss=0.31216, val_loss=0.30804
    Epoch 20: train_loss=0.30944, val_loss=0.30292


KeyboardInterrupt: 

In [ ]:
import torch
import numpy as np
import joblib
from PIL import Image
import clip
from sentence_transformers import SentenceTransformer
import torch.nn as nn
from pathlib import Path
import os

# Define the base project directory (assuming it's already set or in a predictable location)
# This needs to match the os.chdir from the setup cells.
BASE_PROJECT_PATH = Path('/content/drive/MyDrive/Amazon-ML-Challenge-2025')

# Define the device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device for prediction: {DEVICE}")

# =========================
# LOAD MODELS + SCALERS
# =========================

# Helper function to load and verify files
def load_and_verify(filepath, loader_func):
    print(f"Attempting to load from: {filepath}")
    if not filepath.exists():
        print(f"ERROR: File does NOT exist at: {filepath}")
        raise FileNotFoundError(f"File not found: {filepath}")
    else:
        print(f"File found: {filepath}")
        return loader_func(filepath)

# Load scalers using the explicit path and verify
text_scaler_path = BASE_PROJECT_PATH / "text_scaler.pkl"
text_scaler = load_and_verify(text_scaler_path, joblib.load)

image_scaler_path = BASE_PROJECT_PATH / "image_scaler.pkl"
image_scaler = load_and_verify(image_scaler_path, joblib.load)

other_scaler_path = BASE_PROJECT_PATH / "other_scaler.pkl"
other_scaler = load_and_verify(other_scaler_path, joblib.load)

# Define model class (same as training)
class MultimodalFusionMLP(nn.Module):
    def __init__(self, text_dim, image_dim, other_dim, hidden_dim=512, dropout=0.3):
        super().__init__()

        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.other_encoder = nn.Sequential(
            nn.Linear(other_dim, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(dropout * 0.5)
        )

        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)

        # Fusion layers to combine all encoded parts
        # This block MUST match the training model's definition exactly
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 64, hidden_dim * 2), nn.LayerNorm(hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout * 0.7),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, text_emb, image_emb, other_emb):
        text_enc = self.text_encoder(text_emb)
        image_enc = self.image_encoder(image_emb)
        other_enc = self.other_encoder(other_emb)

        attended, _ = self.attention(text_enc.unsqueeze(1), image_enc.unsqueeze(1), image_enc.unsqueeze(1))
        attended = attended.squeeze(1)

        fused = torch.cat([attended, image_enc, other_enc], dim=1)
        return self.fusion(fused)


# Load trained model
model = MultimodalFusionMLP(
    text_dim=384,
    image_dim=512,
    other_dim=other_scaler.n_features_in_ # Use n_features_in_ from the loaded scaler
).to(DEVICE) # <--- Move model to device

# Load the model state dictionary using the explicit path and verify
model_path = BASE_PROJECT_PATH / "model_fold_1.pth"
model.load_state_dict(load_and_verify(model_path, lambda p: torch.load(p, map_location=DEVICE))) # <--- map_location=DEVICE
model.eval()

# =========================
# LOAD EMBEDDING MODELS
# =========================

# Text model
text_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE) # <--- Specify device

# Image model (CLIP)
clip_model, preprocess = clip.load("ViT-B/32", device=DEVICE) # <--- Specify device

# =========================
# PREDICTION FUNCTION
# =========================

def predict_price_from_input(image_path, text_input, other_features):

    # ▒ TEXT → EMBEDDING
    text_emb = text_model.encode(text_input)

    # ▒ IMAGE → EMBEDDING
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        image_emb = clip_model.encode_image(image).cpu().numpy().flatten()

    # ▒ OTHER FEATURES
    # other_features must have 87 dimensions (quantity + one-hot unit + one-hot brand)
    # For now, we'll create a dummy array with the correct size.
    # The first element is quantity, rest are for unit/brand one-hot encodings.
    other_features_vector = np.zeros(other_scaler.n_features_in_, dtype=np.float32)
    # Example: assign a dummy quantity (e.g., 1.0) to the first feature
    other_features_vector[0] = 1.0 # Placeholder for quantity

    # Reshape
    text_emb = text_emb.reshape(1, -1)
    image_emb = image_emb.reshape(1, -1)
    other_features_vector = other_features_vector.reshape(1, -1)

    # ▒ SCALE
    text_scaled = text_scaler.transform(text_emb)
    image_scaled = image_scaler.transform(image_emb)
    other_scaled = other_scaler.transform(other_features_vector)

    # Convert to tensor and move to device
    text_t = torch.FloatTensor(text_scaled).to(DEVICE)
    image_t = torch.FloatTensor(image_scaled).to(DEVICE)
    other_t = torch.FloatTensor(other_scaled).to(DEVICE)

    # ▒ PREDICT
    with torch.no_grad():
        pred_log = model(text_t, image_t, other_t).item()

    price = np.expm1(pred_log)

    return price*10


# =========================
# USER INPUT
# =========================

image_path = input("Enter the path to the product image: ") # Modified to directly input path

print("Selected file:", image_path)
text_input = input("Enter product description: ")

# Example: adjust based on your dataset. The 'other_features' variable here is not directly used
# in the predict_price_from_input function anymore, but rather derived from a dummy for now.
# In a real scenario, you would need to extract quantity, unit, and brand from your input
# and convert them into an 87-dimensional vector to pass to the function.
# For now, the predict_price_from_input function uses a placeholder 87-dim array.
# (example: brand_id, category_id, quantity — change this!)

# =========================
# OUTPUT
# =========================

# Call the prediction function without passing the 'other_features' example array directly
# as it's now handled internally with a placeholder.
price = predict_price_from_input(image_path, text_input, None) # Pass None or an empty list if not used

print("\n▓ Predicted Price:", price)


Using device for prediction: cpu
Attempting to load from: /content/drive/MyDrive/Amazon-ML-Challenge-2025/text_scaler.pkl
File found: /content/drive/MyDrive/Amazon-ML-Challenge-2025/text_scaler.pkl
Attempting to load from: /content/drive/MyDrive/Amazon-ML-Challenge-2025/image_scaler.pkl
File found: /content/drive/MyDrive/Amazon-ML-Challenge-2025/image_scaler.pkl
Attempting to load from: /content/drive/MyDrive/Amazon-ML-Challenge-2025/other_scaler.pkl
File found: /content/drive/MyDrive/Amazon-ML-Challenge-2025/other_scaler.pkl
Attempting to load from: /content/drive/MyDrive/Amazon-ML-Challenge-2025/model_fold_1.pth
File found: /content/drive/MyDrive/Amazon-ML-Challenge-2025/model_fold_1.pth


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 306MiB/s]


Enter the path to the product image: /content/drive/MyDrive/Amazon-ML-Challenge-2025/price/flowerL.jpg
Selected file: /content/drive/MyDrive/Amazon-ML-Challenge-2025/price/flowerL.jpg
Enter product description: flower   "Item Name: KaBloom Flowers - Exotic Perla White Orchid Bouquet of 10 White Orchids Without Vase - Gift for Birthday, Get Well, Easter, Valentine, Mother’s Day Fresh Flowers Bullet Point 1: Composition: The bloom collection includes fresh White Orchids Without Vase. We ship in a specially designed box that protects the flowers during transit. Bullet Point 2: Fresh Flowers: Our fresh flower bouquets are shipped in their bud stage, ensuring easier transport and longer-lasting blooms. The flowers will fully open within 2–3 days. Bullet Point 3: Gift Note: To ensure your gift inlcudes your information for the recipient, At checkout, please check off ""THIS IS A GIFT"" or ""ADD A GIFT RECEIPT"" and please include your name in the message. Bullet Point 4: Care and Handling: T

In [ ]:
print('Installing FFmpeg...')
!apt-get update -qq && apt-get install -y ffmpeg

print('Uninstalling incompatible packages...')
!pip uninstall -y torchcodec sentence-transformers

print('Reinstalling sentence-transformers to ensure compatibility...')
!pip install sentence-transformers

print('Installation/Reinstallation complete. Please re-run the previous prediction cell.')

Installing FFmpeg...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
Uninstalling incompatible packages...
Found existing installation: torchcodec 0.10.0
Uninstalling torchcodec-0.10.0:
  Successfully uninstalled torchcodec-0.10.0
Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Reinstalling sentence-transformers to ensure compatibility...
  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.5.1-py3-none-any.whl (588 kB)
Installation/Reinstallation complete. Plea

In [ ]:
# ============================================================
# LAST CELL — Flask + Cloudflare Tunnel + Inline UI
# ============================================================

!pip install flask flask-cors -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared 2>/dev/null || true
!chmod +x cloudflared 2>/dev/null || true

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading, tempfile, os, subprocess, time, re
import numpy as np
from PIL import Image
import torch
from IPython.display import display, HTML

app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST', 'OPTIONS'])
def predict():
    if request.method == 'OPTIONS':
        from flask import make_response
        r = make_response()
        r.headers['Access-Control-Allow-Origin'] = '*'
        r.headers['Access-Control-Allow-Methods'] = 'POST, OPTIONS'
        r.headers['Access-Control-Allow-Headers'] = 'Content-Type'
        return r, 200
    try:
        image_file = request.files['image']
        text       = request.form.get('text', '')
        quantity   = float(request.form.get('quantity', 1.0))

        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
            image_file.save(tmp.name)
            tmp_path = tmp.name

        text_emb  = text_model.encode(text).reshape(1, -1)
        img       = preprocess(Image.open(tmp_path)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_emb = clip_model.encode_image(img).cpu().numpy().reshape(1, -1)

        other = np.zeros((1, other_scaler.n_features_in_), dtype=np.float32)
        other[0, 0] = quantity

        t = torch.FloatTensor(text_scaler.transform(text_emb)).to(DEVICE)
        i = torch.FloatTensor(image_scaler.transform(image_emb)).to(DEVICE)
        o = torch.FloatTensor(other_scaler.transform(other)).to(DEVICE)

        with torch.no_grad():
            price = np.expm1(model(t, i, o).item()) * 100

        os.unlink(tmp_path)
        return jsonify({"predicted_price": round(float(price), 2)})

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# ── Start Flask ───────────────────────────────────────────────
def run_flask():
    import logging
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)
    app.run(port=5000, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# ── Start Cloudflare tunnel ───────────────────────────────────
cf = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True
)

TUNNEL_URL = ""
for line in cf.stdout:
    m = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if m:
        TUNNEL_URL = m.group(0)
        break

print(f"Tunnel ready: {TUNNEL_URL}")

# ── Render UI inline ──────────────────────────────────────────
html = f"""
<style>
  #pv-app * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', sans-serif; }}
  #pv-app {{ max-width: 680px; padding: 24px; background: #0f1117; border-radius: 16px; color: #f0f0f8; }}
  #pv-app h1 {{ font-size: 24px; font-weight: 700; margin-bottom: 4px; color: #6ee7b7; }}
  #pv-app p.sub {{ font-size: 13px; color: #8888aa; margin-bottom: 20px; }}
  .pv-card {{ background: #1a1a26; border: 1px solid #ffffff15; border-radius: 12px; padding: 16px; margin-bottom: 12px; }}
  .pv-label {{ font-size: 11px; font-weight: 600; text-transform: uppercase; letter-spacing: .06em; color: #8888aa; margin-bottom: 10px; }}
  .pv-drop {{ border: 1.5px dashed #ffffff20; border-radius: 10px; padding: 28px; text-align: center; cursor: pointer; position: relative; transition: border-color .2s; }}
  .pv-drop:hover {{ border-color: #6ee7b7; background: #6ee7b708; }}
  .pv-drop input {{ position: absolute; inset: 0; opacity: 0; cursor: pointer; font-size: 0; }}
  .pv-drop .di {{ font-size: 32px; margin-bottom: 8px; }}
  .pv-drop .dl {{ font-size: 13px; color: #8888aa; }}
  #pv-prev {{ max-height: 160px; max-width: 100%; border-radius: 8px; margin-top: 10px; display: none; }}
  #pv-fname {{ font-size: 12px; color: #6ee7b7; margin-top: 6px; display: none; }}
  #pv-app textarea {{ width: 100%; min-height: 90px; background: #12121a; border: 1px solid #ffffff15; border-radius: 8px; color: #f0f0f8; font-size: 13px; padding: 10px 12px; resize: vertical; outline: none; }}
  #pv-app textarea:focus {{ border-color: #6ee7b750; }}
  #pv-app textarea::placeholder {{ color: #44445a; }}
  .pv-grid {{ display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px; }}
  .pv-field label {{ display: block; font-size: 11px; color: #8888aa; margin-bottom: 5px; }}
  .pv-field input, .pv-field select {{ width: 100%; background: #12121a; border: 1px solid #ffffff15; border-radius: 8px; color: #f0f0f8; font-size: 13px; padding: 8px 10px; outline: none; }}
  #pv-btn {{ width: 100%; margin-top: 14px; padding: 13px; background: #6ee7b7; border: none; border-radius: 10px; color: #0f1117; font-size: 15px; font-weight: 700; cursor: pointer; transition: opacity .2s; }}
  #pv-btn:hover {{ opacity: .85; }}
  #pv-btn:disabled {{ opacity: .35; cursor: not-allowed; }}
  #pv-result {{ display: none; margin-top: 14px; background: #0d1f18; border: 1px solid #6ee7b730; border-radius: 12px; padding: 24px; text-align: center; }}
  #pv-result .rl {{ font-size: 11px; font-weight: 600; letter-spacing: .1em; text-transform: uppercase; color: #6ee7b7; margin-bottom: 8px; }}
  #pv-result .rp {{ font-size: 48px; font-weight: 800; color: #6ee7b7; line-height: 1; }}
  #pv-result .rn {{ font-size: 12px; color: #8888aa; margin-top: 6px; }}
  #pv-error {{ display: none; margin-top: 10px; background: #1a0d0d; border: 1px solid #f8717130; border-radius: 8px; padding: 10px 14px; font-size: 13px; color: #f87171; }}
  .pv-spin {{ width: 16px; height: 16px; border: 2px solid #0f111740; border-top-color: #0f1117; border-radius: 50%; animation: pvspin .6s linear infinite; display: inline-block; vertical-align: middle; margin-right: 6px; }}
  @keyframes pvspin {{ to {{ transform: rotate(360deg); }} }}
</style>

<div id="pv-app">
  <h1> Price Predictor</h1>
  <p class="sub">Amazon ML Challenge 2025 &mdash; Multimodal Fusion Model</p>

  <div class="pv-card">
    <div class="pv-label">Product Image</div>
    <div class="pv-drop" id="pv-drop">
      <input type="file" id="pv-img" accept="image/*">
      <div class="di">&#128247;</div>
      <div class="dl">Click to upload or drag &amp; drop</div>
      <img id="pv-prev" alt="preview">
      <div id="pv-fname"></div>
    </div>
  </div>

  <div class="pv-card">
    <div class="pv-label">Product Description</div>
    <textarea id="pv-txt" placeholder="e.g. Sony WH-1000XM5 wireless headphones, 30hr battery, ANC, USB-C..."></textarea>
  </div>

  <div class="pv-card">
    <div class="pv-label">Optional Details</div>
    <div class="pv-grid">
      <div class="pv-field"><label>Quantity</label><input type="number" id="pv-qty" value="1" min="0" step="0.1"></div>
      <div class="pv-field"><label>Unit</label>
        <select id="pv-unit">
          <option value="">none</option><option>piece</option><option>pack</option>
          <option>set</option><option>kg</option><option>g</option><option>litre</option>
        </select>
      </div>
      <div class="pv-field"><label>Brand</label><input type="text" id="pv-brand" placeholder="e.g. Samsung"></div>
    </div>
  </div>

  <button id="pv-btn" onclick="pvPredict()">&#10022; Predict Price</button>
  <div id="pv-error"></div>
  <div id="pv-result">
    <div class="rl">Estimated Price</div>
    <div class="rp" id="pv-price">&#8377;0</div>
    <div class="rn">CLIP + SentenceTransformer + MLP Fusion</div>
  </div>
</div>

<script>
(function() {{
  const API = "{TUNNEL_URL}/predict";

  const imgInput = document.getElementById('pv-img');
  imgInput.addEventListener('change', e => {{
    const f = e.target.files[0]; if (!f) return;
    document.getElementById('pv-prev').src = URL.createObjectURL(f);
    document.getElementById('pv-prev').style.display = 'block';
    document.getElementById('pv-fname').textContent = '✓ ' + f.name;
    document.getElementById('pv-fname').style.display = 'block';
  }});

  const drop = document.getElementById('pv-drop');
  drop.addEventListener('dragover', e => {{ e.preventDefault(); drop.style.borderColor='#6ee7b7'; }});
  drop.addEventListener('dragleave', () => {{ drop.style.borderColor=''; }});
  drop.addEventListener('drop', e => {{
    e.preventDefault(); drop.style.borderColor='';
    const f = e.dataTransfer.files[0];
    if (f && f.type.startsWith('image/')) {{
      imgInput.files = e.dataTransfer.files;
      imgInput.dispatchEvent(new Event('change'));
    }}
  }});

  window.pvPredict = async function() {{
    const file = imgInput.files[0];
    const text = document.getElementById('pv-txt').value.trim();
    const err  = document.getElementById('pv-error');
    const res  = document.getElementById('pv-result');
    const btn  = document.getElementById('pv-btn');

    err.style.display = 'none';
    res.style.display = 'none';

    if (!file) {{ err.textContent = 'Please upload a product image.'; err.style.display='block'; return; }}
    if (!text) {{ err.textContent = 'Please enter a product description.'; err.style.display='block'; return; }}

    btn.disabled = true;
    btn.innerHTML = '<span class="pv-spin"></span> Analyzing...';

    const fd = new FormData();
    fd.append('image', file);
    fd.append('text', text);
    fd.append('quantity', document.getElementById('pv-qty').value || '1');
    fd.append('unit', document.getElementById('pv-unit').value || '');
    fd.append('brand', document.getElementById('pv-brand').value || '');

    try {{
      const r    = await fetch(API, {{ method: 'POST', body: fd }});
      const data = await r.json();
      if (!r.ok) throw new Error(data.error || 'Server error');
      const fmt = new Intl.NumberFormat('en-IN', {{ maximumFractionDigits: 2 }}).format(data.predicted_price);
      document.getElementById('pv-price').textContent = '₹' + fmt;
      res.style.display = 'block';
    }} catch(e) {{
      err.textContent = e.message.includes('fetch') ? 'Cannot reach backend. Is the tunnel running?' : e.message;
      err.style.display = 'block';
    }} finally {{
      btn.disabled = false;
      btn.innerHTML = '&#10022; Predict Price';
    }}
  }};
}})();
</script>
"""

display(HTML(html))

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Tunnel ready: https://urgent-documentary-looks-zope.trycloudflare.com


In [ ]:
!ls

 cloudflared			 model_fold_3.pth   test_images
'Copy of other_scaler (1).pkl'	 model_fold_4.pth   test_out.csv
'Copy of other_scaler.pkl'	 model_fold_5.pth   text_scaler.pkl
'Documentation Template.md'	 optimised.py	   'train (1).gsheet'
 embeddings			 other_scaler.pkl   train.csv
 embeddings_medium		 price		    train.csv.zip
 image_scaler.pkl		 README.md	    train.gsheet
 __MACOSX			 src		    train_images
 model_fold_1.pth		 test.csv	    venv
 model_fold_2.pth		 test.csv.zip


In [ ]:
%cd /content/drive/MyDrive/Amazon-ML-Challenge-2025/src

/content/drive/MyDrive/Amazon-ML-Challenge-2025/src


In [ ]:
!python app.py